In [ ]:
# import toml
# from pathlib import Path
# config_path = Path("../../../../configuration/asim_configuration")
# input_config = toml.load(config_path/ "input_configuration.toml")
# summary_config = toml.load(config_path/ "summary_configuration.toml")

Free Parking Model predicts the availability of free parking at a person’s workplace. It is applied for people who work in zones that have parking charges

- only workers that have a usual work location outside their home and that is in a zone that has paid parking
    - PRKCST > 0
    - workers with usual work location

In [3]:
import pandas as pd
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

In [4]:
# read data
person = util.get_person_data(summary_config, uncloned=False)

per_data = person.\
    join(util.get_landuse_data(summary_config).select(['zone_id', 'TAZ', 'county_id']), how="left",left_on='home_zone_id',right_on='zone_id').\
    to_pandas()
per_data = per_data.loc[per_data['pemploy']<3]

- number of people who work in zones that have parking charges

In [5]:
per_data.groupby(['source'])['person_weight'].sum().reset_index()

,source,person_weight
0,model,1.982128e+06
1,survey,2.301793e+06


## Work From Home

In [6]:
df_plot = per_data.groupby(['source','work_from_home'])['person_weight'].sum().reset_index()

df_plot['percentage'] = df_plot.groupby(['source'], group_keys=False)['person_weight']. \
    apply(lambda x: x / float(x.sum()))

fig = px.bar(df_plot.sort_values(by=['source']), x="work_from_home", y="percentage", color="source",
             barmode="group",
             title="Work From Home")
fig.update_layout(height=400, width=700, font=dict(size=11), yaxis_tickformat = '.0%')
fig.show()

In [7]:
# auto ownership in Income groups
def plot_segments(df:pd.DataFrame, var:str, title_cat:str,sub_name:str):
    # print(f"n=\n"
    #       f"{df.loc[df['source']=='model results',var].value_counts()[df[var].sort_values().unique()]}")
    df_plot = df.groupby(['source',var,'work_from_home'])['person_weight'].sum().reset_index()
    df_plot['percentage'] = df_plot.groupby(['source',var], group_keys=False)['person_weight'].\
        apply(lambda x: 100 * x / float(x.sum()))

    fig = px.bar(df_plot, x="work_from_home", y="percentage", color="source",
                 facet_col=var, barmode="group",
                 title="Work From Home by "+ title_cat)
    fig.for_each_annotation(lambda a: a.update(text = sub_name + "=<br>" + a.text.split("=")[-1]))
    fig.update_xaxes(title_text="n of work from home")
    fig.update_layout(height=400, width=800, font=dict(size=11))
    fig.show()

In [8]:
plot_segments(per_data,'county_id','County', 'County')